# Kadanoff-Baym equations of motion

$$
\left[i \partial_z - h(z) \right]G(z,z^\prime) - \left[\Sigma * G\right](z,z^\prime) = \delta_C(z,z^\prime) \\
\left[-i \partial_z^\prime - h(z^\prime)\right]G(z,z^\prime) - \left[G * \Sigma\right](z,z^\prime)= \delta_C(z,z^\prime)
$$

In [ ]:
using KadanoffBaym
include("../src/wigner.jl") # Wigner transform

import NCA
using Interpolations, AMR
include("../src/utils.jl") # trapz

import SpecialFunctions: erf, erfc
import LinearAlgebra: diag

using Plots
default(fmt = :png)

## Generating a gaussian pulse

Consider a gaussian pulse
$$\xi(t) = \left(\frac{\Omega^2}{2 \pi}\right)^{1/4}\exp{\left(-\frac{\Omega^2}{4}(t-t_c)^2\right)}$$

Consider a modulator cavity given by the triple $(S_M, L_M, H_M)$

for which $L_M = \lambda(t) a$ where $\lambda(t) \triangleq \frac{\xi(t)}{\sqrt{\int_t^\infty \textrm{d}s |\xi(s)|^2}}$

In [ ]:
# Gaussian pulse parameters
const tc = 0.3  # peak arrival time
const Ω = 20.0 # frequency bandwidth of the pulse

function ξ(t)
    return (Ω^2 / (2π))^(1/4) * exp(-(t - tc)^2 * (Ω^2 / 4))
end

# Pump and loss terms
const Γ = 1.
λ(t::Float64) = iszero(Γ) ? 0.0 : Γ * ξ(t) / sqrt(0.5 * erfc((t - tc) * Ω / sqrt(2)))
γ(t::Float64) = iszero(Γ) ? 0.0 : Γ * ξ(t)

# Photon field Hamiltonian
h(t::Float64) = 0.01 - 0.5im * (λ(t) + γ(t))

# Artificial width of the peak
Σ = 0.1

# Initial field occupation
n₀ = 0.0 

# Initial conditions
G₀ᵞL = GreenFunction(zeros(ComplexF64, 1, 1, 1, 1), Lesser)
G₀ᵞG = GreenFunction(zeros(ComplexF64, 1, 1, 1, 1), Greater)
G₀ᵞL[1,1,1,1] = -1.0im * n₀
G₀ᵞG[1,1,1,1] = -1.0im * (1 + n₀)

# Pack them
u₀ = [G₀ᵞL, G₀ᵞG]; # u0 = VectorOfArray([Lesser0, Greater0]);

# Define commutator
⋊(a,b) = a * b - b * adjoint(a)

function f(u, p, t1, t2)
    t = times[t1]
    GL, GG = u[1], u[2]
    
    rhsL = -1.0im * ((h(t) + 1.0im * γ(t) - 1.0im * Σ) * GL[t1,t2])
    rhsG = -1.0im * ((h(t) + 1.0im * γ(t) - 1.0im * Σ) * GG[t1,t2])
    return [rhsL, rhsG] # VectorOfArray([rhsL, rhsG])
end

function f_diag(u, p, t1, t2)
    t = times[t1]
    GL, GG = u[1], u[2]
    
    rhsL = -1.0im * (h(t) ⋊ GL[t1,t2] + 1.0im * γ(t) * (GL[t1,t2] + GG[t1,t2]))
    rhsG = -1.0im * (h(t) ⋊ GG[t1,t2] + 1.0im * λ(t) * (GG[t1,t2] - GL[t1,t2]) + 2.0im * γ(t) * GG[t1,t2])
    return [rhsL, rhsG] # VectorOfArray([rhsL, rhsG])
end

In [ ]:
# ODE problem is defined by the rhs, initial condition and time span
tspan = (0.0,1.0)
prob = ODEProblem(f, u₀, tspan, f_diag=f_diag)

# Algorithm to timestep is the (Kadanoff-Baym) ABM43 (only really this one exists)
alg = KB{ABM43}()

# This algorithm requires a fixed time
dt = 0.005

times = collect(first(tspan):dt:last(tspan)+dt)

# @assert sum((ξ.(times).^2)) * dt ≈ 1
sol = solve(prob, alg, dt);

GᵞL = sol.u[1][:,:,1,1];
GᵞG = sol.u[2][:,:,1,1];

#### Plotting

In [ ]:
plot(times, λ.(times), label="Loss term")
plot!(times, γ.(times), label="Pump term")
plot!(twinx(), times, ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot, xlabel=L"t")

In [ ]:
times = first.(sol.t[:,1])
plot(times, GᵞL[:,:,1,1] |> diag .|> imag .|> -, label="Lesser")
plot!(times, GᵞG[:,:,1,1] |> diag .|> imag .|> -, label="Greater")
plot!(twinx(), times, ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot, xlabel=L"t")

In [ ]:
# heatmap(times, times, -imag(GᵞG - GᵞL))

# Time-depedent spectral function
ρ = wigner_transform(GᵞG - GᵞL; Nf=4096*2);
wigner_plot!(real(ρ), times)#; ylims=(-10,10))

## Initial correlations

### Equilibrium state

In [ ]:
# Solve the problem in equilibrium
model = NCA.PhotonAssistedModel(
    V₀=sqrt(0.2),
    ε₀f=-0.5, 
    Γ₀=sqrt(0.00), 
    ω₀=0.02, 
    δ₀=0.01)

nca = NCA.FreqNCA(
    model=model,
    β=5e2,
    mesh=NCA.LinearMesh(-6.0,6.0,2^16),
    eps=1e-9)

imΣ = NCA.solve!(nca; project=false);

In [ ]:
# Calculate spectral functions
model.λ₀ = 0.
Aᶜ = nca.data.A_c
Aᵇ, Aᶠ = NCA.slave_particle_spkf(nca, imΣ);
Aᶠ[Aᶠ .< nca.eps] .= 0
Aᵇ[Aᵇ .< nca.eps] .= 0

# λ₀ shift
boltz = NCA.bf.(nca.mesh, nca.β)

# Shift the original mesh
mesh_shift = nca.mesh.mesh .- Float64(-1/nca.β * log(boltz' * (2*Aᶠ + Aᵇ) * nca.mesh.dx));

In [ ]:
# Find optimal mesh
Aᶜ_interp = LinearInterpolation(mesh_shift, Aᶜ);
Aᶠ_interp = LinearInterpolation(mesh_shift, Aᶠ);
Aᵇ_interp = LinearInterpolation(mesh_shift, Aᵇ);

mesh, _ = sample_function(x -> Aᵇ_interp(x) + Aᶠ_interp(x), mesh_shift[1]:0.1:mesh_shift[end]; tol=1e-3);

∫dω(f) = trapz(mesh, f) / (2π)
# plot(mesh, Aᶠ_interp.(mesh))
# plot!(mesh, Aᵇ_interp.(mesh))

In [ ]:
# Equilibrium retarded Green functions
GᶜR = LinearInterpolation(mesh_shift, NCA.kramerskronig(nca.mesh, π * Aᶜ, :real) + 1im * π * Aᶜ)(mesh);
GᶠR = LinearInterpolation(mesh_shift, NCA.kramerskronig(nca.mesh, π * Aᶠ, :real) + 1im * π * Aᶠ)(mesh);
GᵇR = LinearInterpolation(mesh_shift, NCA.kramerskronig(nca.mesh, π * Aᵇ, :real) + 1im * π * Aᵇ)(mesh);

In [ ]:
plot(mesh, imag(GᵇR))

In [ ]:
# Assert GFs and As are normalized!
println(∫dω(Aᶜ_interp.(mesh)) * 2π)
println(∫dω(GᶜR - conj(GᶜR)) |> imag) 
println(∫dω(Aᶠ_interp.(mesh)) * 2π)
println(∫dω(GᶠR - conj(GᶠR)) |> imag) 
println(∫dω(Aᵇ_interp.(mesh)) * 2π)
println(∫dω(GᵇR - conj(GᵇR)) |> imag) 

#### Initial conditions

$$
G^<(t_0, t_0) = \mathcal{G}(0, 0^+) = \frac{i}{\beta} \sum_n e^{+i \omega_n 0^+} \mathcal{G}(i \omega_n) = \mp i \int_{-\infty}^{+\infty} \frac{\textrm{d} \omega}{2\pi} f(\omega) A(\omega)
$$
$$
G^>(t_0, t_0) = \mathcal{G}(0^+, 0) = \frac{i}{\beta} \sum_n e^{-i \omega_n 0^+} \mathcal{G}(i \omega_n) = - i \int_{-\infty}^{+\infty} \frac{\textrm{d} \omega}{2\pi} \left(1 \pm f(\omega)\right) A(\omega)
$$

In [ ]:
boltz = NCA.bf.(mesh, nca.β) .|> Float64; boltz[boltz .== Inf] .= 0.0
bose = NCA.nb.(mesh, nca.β); bose[bose .== Inf] .= 0.0
fermi = NCA.nf.(mesh, nca.β)

G₀ᶜL = +1.0im * ∫dω(fermi .* Aᶜ_interp.(mesh)) * (2π)
G₀ᶜL = GreenFunction(G₀ᶜL * ones(1,1,1,1), Lesser)

G₀ᶜG = -1.0im * (1 - ∫dω(fermi .* Aᶜ_interp.(mesh)) * (2π))
G₀ᶜG = GreenFunction(G₀ᶜG * ones(1,1,1,1), Greater)

G₀ᶠL = +1.0im * ∫dω(boltz .* Aᶠ_interp.(mesh)) * (2π)
G₀ᶠL = GreenFunction(G₀ᶠL * ones(1,1,1,1), Lesser)

G₀ᶠG = -1.0im * (1 - ∫dω(boltz .* Aᶠ_interp.(mesh)) * (2π))
G₀ᶠG = GreenFunction(G₀ᶠG * ones(1,1,1,1), Greater)

G₀ᵇL = -1.0im * ∫dω(boltz .* Aᵇ_interp.(mesh)) * (2π)
G₀ᵇL = GreenFunction(G₀ᵇL * ones(1,1,1,1), Lesser)

G₀ᵇG = -1.0im * (1 + ∫dω(boltz .* Aᵇ_interp.(mesh)) * (2π))
G₀ᵇG = GreenFunction(G₀ᵇG * ones(1,1,1,1), Greater)

u₀ = [G₀ᶜL, G₀ᶜG, G₀ᶠL, G₀ᶠG, G₀ᵇL, G₀ᵇG];
u₀_lin = [GᶜR, conj(GᶜR), GᶠR, conj(GᶠR), GᵇR, conj(GᵇR)];

### ODEs

$$
i \partial_t G^\gtrless(t,t^\prime) =
h(t) G^\gtrless(t,t^\prime)
+ \int_{t_0}^t \textrm{d}\bar{t} \left[\Sigma^>(t,\bar{t}) - \Sigma^<(t,\bar{t})\right] G^\gtrless(\bar{t},t^\prime)
- \int_{t_0}^{t^\prime} \textrm{d}\bar{t} \Sigma^\gtrless(t, \bar{t}) \left[G^>(\bar{t},t^\prime) - G^<(\bar{t},t^\prime)\right]
- i \int_0^\beta \textrm{d}\tau \Sigma^\rceil(t,\tau) G^\lceil(\tau,t^\prime)
$$
$$
-i \partial_{t^\prime} G^\gtrless(t,t^\prime) =
G^\gtrless(t,t^\prime) h(t)^\dagger
+ \int_{t_0}^{t} \textrm{d}\bar{t} \left[G^>(t,\bar{t}) - G^<(t,\bar{t})\right] \Sigma^\gtrless(\bar{t},t^\prime)
- \int_{t_0}^{t^\prime} \textrm{d}\bar{t} G^\gtrless(t,\bar{t}) \left[\Sigma^>(\bar{t},t^\prime) - \Sigma^<(\bar{t},t^\prime)\right] 
- i \int_0^\beta \textrm{d}\tau G^\rceil(t,\tau)\Sigma^\lceil(\tau,t^\prime) 
$$

In [ ]:
σ = 0.5
Γ₀ = 0.1
V₀ = model.V₀

"""
Vertical stepping
"""
function f_vert(u, u_lin, t1, t2)
    # Integral 
    ∫dt(i,j,A::Function,B::GreenFunction) = sign(j - i) * sum(A(t1,t) * B[t,t2] for t=min(i,j):max(i,j)) * dt
    
    # unpack Green functions
    GᶜL, GᶜG, GᶠL, GᶠG, GᵇL, GᵇG, = u
    Gᶜ˥p, Gᶜ˥m, Gᶠ˥p, Gᶠ˥m, Gᵇ˥p, Gᵇ˥m = u_lin

    # c-electron self-energies
    ΣᶜL(i,j) = 0.0
    ΣᶜG(i,j) = 0.0
    Σᶜ˥p = 0.0
    Σᶜ˥m = 0.0
    
    # f-electron self-energies
    ΣᶠL(i,j) = V₀^2 * GᶜL[i,j] * GᵇL[i,j] + Γ₀^2 * GᶜL[i,j] * (GᵞL[i,j] + GᵞG[j,i]) * GᵇL[i,j]
    ΣᶠG(i,j) = V₀^2 * GᶜG[i,j] * GᵇG[i,j] + Γ₀^2 * GᶜG[i,j] * (GᵞL[i,j] + GᵞG[j,i]) * GᵇG[i,j]
    Σᶠ˥p = V₀^2 * Gᶜ˥p[:,t1] .* Gᵇ˥p[:,t1]
    Σᶠ˥m = V₀^2 * Gᶜ˥m[:,t1] .* Gᵇ˥m[:,t1]
    
    # slave boson self-energies
    ΣᵇL(i,j) = V₀^2 * GᶠL[i,j] * GᶜG[j,i] + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i]) * GᶠL[i,j] * GᶜG[j,i] 
    ΣᵇG(i,j) = V₀^2 * GᶠG[i,j] * GᶜL[j,i] + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i]) * GᶠG[i,j] * GᶜL[j,i] 
    Σᵇ˥p = V₀^2 * Gᶠ˥p[:,t1] .* conj(reverse(Gᶜ˥p[:,t1]))
    Σᵇ˥m = V₀^2 * Gᶠ˥m[:,t1] .* conj(reverse(Gᶜ˥m[:,t1]))
    
    dGᶜG = -1.0im * (- 1.0im * σ * GᶜG[t1,t2])
    dGᶜL = -1.0im * (- 1.0im * σ * GᶜL[t1,t2])
    
#     println(size(fermi))
#     println(size(Σᶠ˥p))
#     println(size(conj(reverse(Gᶠ˥m[:,t1]))))
#     println(typeof(model.ε₀f * GᶠL[t1,t2]))
#     println(typeof(∫dt(1,t1,ΣᶠG,GᶠL)))
#     println(typeof(∫dω(fermi .* (Σᶠ˥p .* conj(reverse(Gᶠ˥m[:,t1]))))))
    dGᶠG = -1.0im * (model.ε₀f * GᶠG[t1,t2] + ∫dt(t1,t2,ΣᶠG,GᶠG))
    dGᶠL = -1.0im * (model.ε₀f * GᶠL[t1,t2] + ∫dt(1,t1,ΣᶠG,GᶠL) - ∫dt(1,t2,ΣᶠG,GᶠG)
        .+ ∫dω(fermi .* (Σᶠ˥p .* conj(reverse(Gᶠ˥m[:,t1])) .- Σᶠ˥m .* conj(reverse(Gᶠ˥p[:,t1])))))
    
    dGᵇG = -1.0im * (0.0 * GᵇG[t1,t2] + ∫dt(t1,t2,ΣᵇG,GᵇG))
    dGᵇL = -1.0im * (0.0 * GᵇL[t1,t2] + ∫dt(1,t1,ΣᵇG,GᵇL) - ∫dt(1,t2,ΣᵇG,GᵇG)
        .- ∫dω(bose .* (Σᵇ˥p .* conj(reverse(Gᵇ˥m[:,t1])) .- Σᵇ˥m .* conj(reverse(Gᵇ˥p[:,t1])))))
    
    return [dGᶜL, dGᶜG, dGᶠL, dGᶠG, dGᵇL, dGᵇG]
end

function f_diag(u, u_lin, t1, t2)
    # Integral 
    ∫dt(i,j,A::Function,B::GreenFunction) = sign(j - i) * sum(A(t1,t) * B[t,t2] for t=min(i,j):max(i,j)) * dt
    ∫dt(i,j,A::GreenFunction,B::Function) = sign(j - i) * sum(A[t1,t] * B(t,t2) for t=min(i,j):max(i,j)) * dt
    
    # unpack Green functions
    GᶜL, GᶜG, GᶠL, GᶠG, GᵇL, GᵇG, = u
    Gᶜ˥p, Gᶜ˥m, Gᶠ˥p, Gᶠ˥m, Gᵇ˥p, Gᵇ˥m = u_lin
    
    # c-electron self-energies
    ΣᶜL(i,j) = 0.0
    ΣᶜG(i,j) = 0.0
    Σᶜ˥p = 0.0
    Σᶜ˥m = 0.0
    
    # f-electron self-energies
    ΣᶠL(i,j) = V₀^2 * GᶜL[i,j] * GᵇL[i,j] + Γ₀^2 * GᶜL[i,j] * (GᵞL[i,j] + GᵞG[j,i]) * GᵇL[i,j]
    ΣᶠG(i,j) = V₀^2 * GᶜG[i,j] * GᵇG[i,j] + Γ₀^2 * GᶜG[i,j] * (GᵞL[i,j] + GᵞG[j,i]) * GᵇG[i,j]
    Σᶠ˥p = V₀^2 * Gᶜ˥p[:,t1] .* Gᵇ˥p[:,t1]
    Σᶠ˥m = V₀^2 * Gᶜ˥m[:,t1] .* Gᵇ˥m[:,t1]
    
    # slave boson self-energies
    ΣᵇL(i,j) = V₀^2 * GᶠL[i,j] * GᶜG[j,i] + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i]) * GᶠL[i,j] * GᶜG[j,i] 
    ΣᵇG(i,j) = V₀^2 * GᶠG[i,j] * GᶜL[j,i] + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i]) * GᶠG[i,j] * GᶜL[j,i] 
    Σᵇ˥p = V₀^2 * Gᶠ˥p[:,t1] .* conj(reverse(Gᶜ˥p[:,t1]))
    Σᵇ˥m = V₀^2 * Gᶠ˥m[:,t1] .* conj(reverse(Gᶜ˥m[:,t1]))
    
    dGᶜG = -1.0im * zero(GᶜG[t1,t2])
    dGᶜL = -1.0im * zero(GᶜL[t1,t2])
    
    dGᶠG = -1.0im * (zero(GᶠG[t1,t2]) + ∫dt(t1,t2,ΣᶠG,GᶠG) - ∫dt(t1,t2,GᶠG,ΣᶠG))
    dGᶠL = -1.0im * (zero(GᶠL[t1,t2]) + ∫dt(1,t1,ΣᶠG,GᶠL) - ∫dt(1,t2,ΣᶠG,GᶠG)
        .+ ∫dω(fermi .* (Σᶠ˥p .* conj(reverse(Gᶠ˥m[:,t1])) - Σᶠ˥m .* conj(reverse(Gᶠ˥p[:,t1]))))
        .- ∫dω(fermi .* (Gᶠ˥m[:,t1] .* conj(reverse(Σᶠ˥p)) - Gᶠ˥p[:,t1] .* conj(reverse(Σᶠ˥m)))))
    
    dGᵇG = -1.0im * (zero(GᵇG[t1,t2]) + ∫dt(t1,t2,ΣᵇG,GᵇG) - ∫dt(t1,t2,GᵇG,ΣᵇG))
    dGᵇL = -1.0im * (zero(GᵇL[t1,t2]) + ∫dt(1,t1,ΣᵇG,GᵇL) - ∫dt(1,t2,ΣᵇG,GᵇG)
        .- ∫dω(bose .* (Σᵇ˥p .* conj(reverse(Gᵇ˥m[:,t1])) - Σᵇ˥m .* conj(reverse(Gᵇ˥p[:,t1]))))
        .+ ∫dω(bose .* (Gᵇ˥m[:,t1] .* conj(reverse(Σᵇ˥p)) - Gᵇ˥p[:,t1] .* conj(reverse(Σᵇ˥m)))))
    
    return [dGᶜL, dGᶜG, dGᶠL, dGᶠG, dGᵇL, dGᵇG]
end

function f_lin(u, u_lin, t1)
    # Integral 
    ∫dt(A::Function,B) = sum(A(t1,t)[1,1] * B[:,t] for t=1:t1) * dt
    
    # unpack Green functions
    GᶜL, GᶜG, GᶠL, GᶠG, GᵇL, GᵇG, = u
    Gᶜ˥p, Gᶜ˥m, Gᶠ˥p, Gᶠ˥m, Gᵇ˥p, Gᵇ˥m = u_lin
    
    # c-electron self-energies
    ΣᶜL(i,j) = 0.0
    ΣᶜG(i,j) = 0.0
    Σᶜ˥p = 0.0
    Σᶜ˥m = 0.0
    
    # f-electron self-energies
    ΣᶠL(i,j) = V₀^2 * GᶜL[i,j] * GᵇL[i,j] + Γ₀^2 * GᶜL[i,j] * (GᵞL[i,j] + GᵞG[j,i]) * GᵇL[i,j]
    ΣᶠG(i,j) = V₀^2 * GᶜG[i,j] * GᵇG[i,j] + Γ₀^2 * GᶜG[i,j] * (GᵞL[i,j] + GᵞG[j,i]) * GᵇG[i,j]
    Σᶠ˥p = V₀^2 * Gᶜ˥p[:,t1] .* Gᵇ˥p[:,t1]
    Σᶠ˥m = V₀^2 * Gᶜ˥m[:,t1] .* Gᵇ˥m[:,t1]
    
    # slave boson self-energies
    ΣᵇL(i,j) = V₀^2 * GᶠL[i,j] * GᶜG[j,i] + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i]) * GᶠL[i,j] * GᶜG[j,i] 
    ΣᵇG(i,j) = V₀^2 * GᶠG[i,j] * GᶜL[j,i] + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i]) * GᶠG[i,j] * GᶜL[j,i] 
    Σᵇ˥p = V₀^2 * Gᶠ˥p[:,t1] .* conj(reverse(Gᶜ˥p[:,t1]))
    Σᵇ˥m = V₀^2 * Gᶠ˥m[:,t1] .* conj(reverse(Gᶜ˥m[:,t1]))

    dGᶜ˥p = -1.0im * (-1.0im * σ * Gᶜ˥p[:,t1]) #+ ∫dt(ζ ΣᶜG, Gᶜ˥p) - ∫dt(ζ ΣᶜL, Gᶜ˥p) + ζ Σᶜ˥p(t,:) .* GᶜR)
    dGᶜ˥m = -1.0im * (-1.0im * σ * Gᶜ˥m[:,t1]) #+ ∫dt(ζ ΣᶜG, Gᶜ˥m) - ∫dt(ζ ΣᶜL, Gᶜ˥m) + ζ Σᶜ˥m(t,:) .* adjoint(GᶜR))
    
    dGᶠ˥p = -1.0im * (+model.ε₀f * Gᶠ˥p[:,t1] + ∫dt(ΣᶠG, Gᶠ˥p) + Σᶠ˥p .* GᶠR) # - ∫dt(ζ ΣᶠL, Gᶠ˥p)
    dGᶠ˥m = -1.0im * (+model.ε₀f * Gᶠ˥m[:,t1] + ∫dt(ΣᶠG, Gᶠ˥p) + Σᶠ˥m .* conj(GᶠR)) # - ∫dt(ζ ΣᶠL, Gᶠ˥m)
    
    dGᵇ˥p = -1.0im * (+0.0 * zero(Gᵇ˥p[:,t1]) + ∫dt(ΣᵇG, Gᶜ˥p) + Σᵇ˥p .* GᵇR) # - ∫dt(ζ ΣᵇL, Gᵇ˥p)
    dGᵇ˥m = -1.0im * (+0.0 * zero(Gᵇ˥m[:,t1]) + ∫dt(ΣᵇG, Gᵇ˥p) + Σᵇ˥m .* conj(GᵇR)) # - ∫dt(ζ ΣᵇL, Gᵇ˥m)
        
    return [dGᶜ˥p, dGᶜ˥m, dGᶠ˥p, dGᶠ˥m, dGᵇ˥p, dGᵇ˥m]
end

In [ ]:
# ODE problem is defined by the rhs, initial condition and time span
tspan = (0.0,1.0)
prob = ODEProblem(f_vert, u₀, tspan; f_diag=f_diag, f_lin=f_lin, u0_lin=u₀_lin)

# Algorithm to timestep is the (Kadanoff-Baym) ABM43 (only really this one exists)
alg = KB{ABM43}()

# This algorithm requires a fixed time
dt = 0.005

times = collect(first(tspan):dt:last(tspan)+dt)

# @assert sum((ξ.(times).^2)) * dt ≈ 1
sol = solve(prob, alg, dt);

In [ ]:
times = first.(sol.t[:,1])

GᶜL = sol.u[1][:,:,1,1];
GᶜG = sol.u[2][:,:,1,1];

GᶠL = sol.u[1][:,:,1,1];
GᶠG = sol.u[2][:,:,1,1];

In [ ]:
heatmap(imag(GᶠG - GᶠL))

In [ ]:
ρ = wigner_transform(GᶠG - GᶠL; Nf=4096*2);
wigner_plot!(real(ρ), times)#; ylims=(-10,10))